In [ ]:
import os
import sys
from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3.12"
os.environ["PYSPARK_DRIVER_PYTHON"] = "/usr/local/bin/python"

spark = (
    SparkSession.builder
    .appName("Workspace MinIO Test")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter-workspace")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config(
        "spark.hadoop.fs.s3a.access.key",
        os.environ["AWS_ACCESS_KEY_ID"],
    )
    .config(
        "spark.hadoop.fs.s3a.secret.key",
        os.environ["AWS_SECRET_ACCESS_KEY"],
    )
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .getOrCreate()
)

print("Driver executable:", sys.executable)
print("PYSPARK_PYTHON:", os.environ.get("PYSPARK_PYTHON"))
print("PYSPARK_DRIVER_PYTHON:", os.environ.get("PYSPARK_DRIVER_PYTHON"))
print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)


In [ ]:
test_df = spark.createDataFrame(
    [
        (1, "Alice"),
        (2, "Bob"),
        (3, "Charlie"),
    ],
    ["id", "name"],
)

test_df.show()


In [ ]:
workspace_path = "s3a://ai-data/workspaces/workspace-dev/outputs/test"

test_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(workspace_path)

print("Write successful:", workspace_path)


In [ ]:
result_df = (
    spark.read
    .format("delta")
    .load(workspace_path)
)

result_df.show()

print("Rows read back:", result_df.count())


In [ ]:
print("Delta table path:", workspace_path)
print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)
print("Workspace MinIO path:", workspace_path)
print("Rows written/read:", result_df.count())
